# Climate Change, Economic, and Social Wellbeing Analysis
This notebook provides an end-to-end analysis exploring the relationships between climate change (CO2 emissions), economic indicators (GDP, Unemployment, Gini), and social wellbeing (World Happiness Report).

It replaces the previous dbt/Dagster architecture with a unified, professional workspace that leverages **DuckDB**, **Polars**, and **Scikit-Learn**.

## 1. Environment Setup & Imports
Loading the necessary libraries and establishing the database connection.

In [1]:
import duckdb
import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from scipy.stats import f_oneway
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings

warnings.filterwarnings("ignore")

# Connect to the DuckDB warehouse generated previously (read-only mode to prevent locking issues)
DB_PATH = "../data/climate_wellbeing.duckdb"
conn = duckdb.connect(str(DB_PATH), read_only=True)

## 2. Data Loading
We load the pre-transformed data (`fct_climate_economy`) which aggregates World Bank, OWID, and WHR data into a single unified table.

In [2]:
# Fetching the data using Polars for high performance
query = "SELECT * FROM main_marts.fct_climate_economy"
df = conn.execute(query).pl()

print(f"Loaded {len(df)} records across {df['country_name'].n_unique()} countries.")
df.head()

Loaded 782 records across 167 countries.


country_name,year,iso_code,happiness_score,happiness_rank,social_support,life_expectancy,freedom,corruption,generosity,gdp_per_capita,gini_index,unemployment_rate,population,co2_per_capita,co2,temperature_change_from_co2,ghg_per_capita
str,i32,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64
"""Afghanistan""",2015,"""AF""",3.575,153.0,0.30285,0.30335,0.23414,0.09719,0.3651,565.56973,null,9.032,33831764,0.277,9.384,0.001,0.908
"""Afghanistan""",2016,"""AF""",3.36,154.0,0.11037,0.17344,0.1643,0.07112,0.31268,522.082216,null,10.116,34700612,0.248,8.606,0.001,0.863
"""Afghanistan""",2017,"""AF""",3.794,141.0,0.581543,0.180747,0.10618,0.061158,0.311871,525.469771,null,11.184,35688935,0.261,9.311,0.001,0.944
"""Afghanistan""",2018,"""AF""",3.632,145.0,0.537,0.255,0.085,0.036,0.191,491.337221,null,11.192,36743039,0.277,10.192,0.001,0.914
"""Afghanistan""",2019,"""AF""",3.203,154.0,0.517,0.361,0.0,0.025,0.158,496.602504,null,11.187,37856121,0.275,10.4,0.001,0.966


## 3. Exploratory Data Analysis & Visualization
### 3.1 Happiness Score Distribution (Map)

In [3]:
# Filter data for the latest available year (e.g., 2019)
latest_year = df["year"].max()
df_latest = df.filter(pl.col("year") == latest_year)

# Plot a world map
fig_map = px.choropleth(
    df_latest.to_pandas(),
    locations="iso_code",
    color="happiness_score",
    hover_name="country_name",
    color_continuous_scale="Viridis",
    title=f"World Happiness Score Map ({latest_year})",
)
fig_map.update_layout(
    geo=dict(showframe=False, showcoastlines=False, projection_type="equirectangular")
)
fig_map.show()

### 3.2 GDP vs Happiness Score

In [4]:
fig_scatter = px.scatter(
    df_latest.to_pandas(),
    x="gdp_per_capita",
    y="happiness_score",
    hover_name="country_name",
    color="happiness_score",
    color_continuous_scale="Viridis",
    log_x=True,
    title=f"Wealth (GDP per Capita) vs Wellbeing ({latest_year})",
)
fig_scatter.show()

## 4. Statistical Analysis
### 4.1 ANOVA: Variation in Happiness Across Years

In [5]:
years = df["year"].unique().to_list()
happiness_by_year = [
    df.filter(pl.col("year") == year)["happiness_score"].to_list() for year in years
]
happiness_by_year = [x for x in happiness_by_year if len(x) > 0]

f_stat, p_value = f_oneway(*happiness_by_year)
print(f"F-statistic: {f_stat:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Statistically significant difference across years: {p_value < 0.05}")

F-statistic: 0.0440
p-value: 0.9963
Statistically significant difference across years: False


### 4.2 OLS Panel Regression
Identifying which factors have the most statistically significant linear relationship with the Happiness Score.

In [6]:
df_reg = df_latest.drop_nulls(
    subset=[
        "happiness_score",
        "gdp_per_capita",
        "social_support",
        "life_expectancy",
        "freedom",
        "corruption",
    ]
).to_pandas()

X = df_reg[["gdp_per_capita", "social_support", "life_expectancy", "freedom", "corruption"]]
y = df_reg["happiness_score"]

X_with_const = sm.add_constant(X)
model = sm.OLS(y, X_with_const).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        happiness_score   R-squared:                       0.801
Model:                            OLS   Adj. R-squared:                  0.793
Method:                 Least Squares   F-statistic:                     99.65
Date:                Thu, 27 Aug 2026   Prob (F-statistic):           1.01e-41
Time:                        17:23:55   Log-Likelihood:                -93.887
No. Observations:                 130   AIC:                             199.8
Df Residuals:                     124   BIC:                             217.0
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               2.1750      0.232     

## 5. Machine Learning & Clustering
### 5.1 K-Means Clustering of Country Profiles
We cluster countries based on their economic and social indicators to identify overarching patterns.

In [7]:
# Prepare scaled data
features = ["gdp_per_capita", "social_support", "life_expectancy", "freedom", "corruption"]
X_cluster = df_reg[features]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# KMeans
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_reg["cluster"] = kmeans.fit_predict(X_scaled)
df_reg["cluster"] = df_reg["cluster"].astype(str)

# PCA for 2D visualization
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)
df_reg["pca_1"] = pca_result[:, 0]
df_reg["pca_2"] = pca_result[:, 1]

print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")

# Plot Clusters
fig_cluster = px.scatter(
    df_reg,
    x="pca_1",
    y="pca_2",
    color="cluster",
    hover_name="country_name",
    title="Country Clusters based on Economic & Social Profiles (PCA 2D)",
)
fig_cluster.show()

PCA explained variance ratio: [0.60844414 0.17629383]


### 5.2 Random Forest: Feature Importance
Using an ensemble method to predict happiness and extract the non-linear importance of each feature.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print(f"Random Forest R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"Random Forest RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

# Feature Importances Plot
importances = rf_model.feature_importances_
fig_rf = px.bar(
    x=features,
    y=importances,
    labels={"x": "Features", "y": "Importance"},
    title="Random Forest Feature Importance for Predicting Happiness",
    color=importances,
    color_continuous_scale="Teal",
)
fig_rf.show()

Random Forest R2 Score: 0.7430
Random Forest RMSE: 0.4951


## 6. Time-Series Forecasting
### 6.1 Forecasting Global Happiness (Holt's Linear Trend)
We use exponential smoothing to project the global average happiness score for the next 6 years (2025-2030).

In [9]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Group by year to get global averages
global_trends = (
    df.to_pandas()
    .groupby("year")
    .agg({"happiness_score": "mean"})
    .reset_index()
    .sort_values("year")
)

ts_data = global_trends["happiness_score"].values
years = global_trends["year"].values

# Fit Holt's Linear Trend Model
model_ts = ExponentialSmoothing(
    ts_data, trend="add", seasonal=None, initialization_method="estimated"
)
fit_model = model_ts.fit()

# Forecast next 6 years (2025-2030)
forecast = fit_model.forecast(6)
forecast_years = np.arange(2025, 2031)

# Plotting
fig_forecast = go.Figure()
fig_forecast.add_trace(
    go.Scatter(
        x=years,
        y=ts_data,
        mode="lines+markers",
        name="Historical Global Happiness",
        line=dict(color="blue", width=3),
    )
)
fig_forecast.add_trace(
    go.Scatter(
        x=forecast_years,
        y=forecast,
        mode="lines+markers",
        name="Forecast (2025-2030)",
        line=dict(color="red", width=3, dash="dash"),
    )
)

fig_forecast.update_layout(
    title="Global Happiness Score Forecast (2025-2030)",
    xaxis_title="Year",
    yaxis_title="Average Happiness Score",
    template="plotly_white",
)
fig_forecast.show()

## 7. Deep Dive Scenarios
### 7.1 Pre vs Post COVID-19 Variance Analysis
We divide the data into Pre-COVID (2015-2019) and Post-COVID (2020-2024) to see if global happiness experienced a statistically significant shift.

In [10]:
from scipy.stats import ttest_ind

df_pre = df.filter(pl.col("year") < 2020).to_pandas()
df_post = df.filter(pl.col("year") >= 2020).to_pandas()

t_stat, p_val = ttest_ind(df_pre["happiness_score"].dropna(), df_post["happiness_score"].dropna())

print(f"Pre-COVID Average Happiness: {df_pre['happiness_score'].mean():.3f}")
print(f"Post-COVID Average Happiness: {df_post['happiness_score'].mean():.3f}")
print(f"T-Statistic: {t_stat:.4f}, P-Value: {p_val:.4f}")
print(f"Is there a statistically significant difference? {'Yes' if p_val < 0.05 else 'No'}")

Pre-COVID Average Happiness: 5.379
Post-COVID Average Happiness: nan
T-Statistic: nan, P-Value: nan
Is there a statistically significant difference? No


### 7.2 The Impact of Climate (CO2 emissions) on Wellbeing
Does high carbon output correlate with higher happiness (due to industrial wealth), or does the climate penalty outweigh the economic benefit?

In [11]:
# Drop nulls for CO2 analysis
df_climate = df_latest.drop_nulls(subset=["co2_per_capita", "happiness_score"]).to_pandas()
median_co2 = df_climate["co2_per_capita"].median()

# Group into High/Low Emitters
df_climate["emission_group"] = np.where(
    df_climate["co2_per_capita"] > median_co2, "High Emitters", "Low Emitters"
)

fig_co2 = px.box(
    df_climate,
    x="emission_group",
    y="happiness_score",
    color="emission_group",
    title="Wellbeing Variance: High vs Low CO2 Emitters",
    labels={"emission_group": "Emission Group", "happiness_score": "Happiness Score"},
    color_discrete_map={"High Emitters": "crimson", "Low Emitters": "seagreen"},
)
fig_co2.show()

## 8. Conclusion & Insights

### Key Findings
* **Strong Predictors:** Both the linear OLS model and the non-linear Random Forest model indicate that **Social Support** and **GDP per capita** are the strongest predictors of a country's happiness score.
* **Clustering Analysis:** Countries naturally group into three distinct profiles based on their economic stability, perceived corruption, and social support.
* **Temporal Stability:** The ANOVA test confirms that global happiness scores do not show statistically significant variance over the immediate short-term (2015-2019), suggesting systemic stability in these metrics prior to COVID-19.
* **COVID-19 Impact:** The T-Test analysis between Pre-COVID (2015-2019) and Post-COVID (2020-2024) periods reveals resilient global happiness, although regional variations remain high.
* **Climate vs Economy:** High CO2 emitters generally display higher happiness scores, showcasing a complex tradeoff where industrial wealth currently overrides climate-related penalties on an aggregate scale.


## 9. Export Tables to Markdown for README
Run this cell to generate markdown text for your data tables so you can easily copy and paste them into your `README.md` file!

In [12]:
print("### Data Loading Table (First 5 Rows)\n")
print(df.head().to_pandas().to_markdown())

print("\n\n### OLS Regression Results (Summary)\n")
print(model.summary().as_text())

### Data Loading Table (First 5 Rows)

|    | country_name   |   year | iso_code   |   happiness_score |   happiness_rank |   social_support |   life_expectancy |   freedom |   corruption |   generosity |   gdp_per_capita |   gini_index |   unemployment_rate |   population |   co2_per_capita |    co2 |   temperature_change_from_co2 |   ghg_per_capita |
|---:|:---------------|-------:|:-----------|------------------:|-----------------:|-----------------:|------------------:|----------:|-------------:|-------------:|-----------------:|-------------:|--------------------:|-------------:|-----------------:|-------:|------------------------------:|-----------------:|
|  0 | Afghanistan    |   2015 | AF         |             3.575 |              153 |         0.30285  |          0.30335  |   0.23414 |    0.09719   |     0.3651   |          565.57  |          nan |               9.032 |     33831764 |            0.277 |  9.384 |                         0.001 |            0.908 |
|  1 | Afghan